# Task 3: running a model locally + optimisation (Ollama, Qwen2.5-1.5B)
Turn on **GPU T4**, then **Run All**. The server is pinned to GPU 0 and everything goes through Ollama's HTTP API.

Produces: speed + VRAM per quantisation (q8_0 vs q4_K_M, optional fp16), VRAM vs context window, KV-cache size from the model config, a concurrency test, executable quality checks, a side-by-side file for YOUR observations, and a 20-user planning calculator.

**Two server phases on purpose:** speed/VRAM are measured with 1 parallel slot (Ollama reserves KV cache per slot, which would inflate VRAM); the server is then restarted with 8 slots for the concurrency test only.

**Download the zip in the last cell before closing Kaggle.**

In [4]:
!sudo apt-get update -qq && sudo apt-get install -y -qq pciutils zstd
!curl -fsSL https://ollama.com/install.sh | sh

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%                                                             17.6%                                                       19.0%####                                                          23.0%#                                           43.7%##########                                        47.7%###################                                  56.0%#####################                               60.3%##########################                           66.3%############################                   77.7%#################################                   77.8%#####################

In [5]:
%%writefile task3_lib.py
"""Shared helpers for Task 3. Talks to Ollama ONLY through its HTTP API (nvidia-smi is used for GPU memory)."""
import json
import os
import subprocess
import time
import urllib.error
import urllib.request

OLLAMA_URL = os.environ.get("OLLAMA_URL", "http://127.0.0.1:11434")
SLEEP = float(os.environ.get("T3_SLEEP", "2"))  # settle time after unloading a model
MODELS = ["qwen2.5:1.5b-instruct-q8_0", "qwen2.5:1.5b-instruct-q4_K_M"]
FP16_MODEL = "qwen2.5:1.5b-instruct-fp16"   # optional third setting (--fp16)

PROMPTS = [
    "Explain the difference between a list and a tuple in Python in 3 concise sentences.",
    "Write a Python function named is_palindrome(s) returning True if s is a palindrome, ignoring spaces, punctuation, and case.",
    "Extract into valid JSON only with exact keys name, role, years_exp: 'Priya Sharma is a Senior ML Engineer with 6 years of experience.'",
    "A bat and a ball cost $1.10 total. The bat costs $1.00 more than the ball. How much does the ball cost? Explain briefly.",
    "Summarize why GPUs are faster than CPUs for matrix multiplication in 3 bullet points.",
    "Find the bug in this Python function and fix it:\ndef average(nums):\n    return sum(nums) / len(nums)",
    "Briefly explain what model quantization (Q8_0 vs Q4_K_M) does to an LLM's memory usage and speed.",
    "Rewrite in formal business English: 'Hey, can u send me the report asap? thx.'",
    "Write a SQL query to find the top 3 highest-paid employees in each department from employees(id, name, department, salary).",
    "Explain the difference between precision and recall in machine learning with a quick example.",
]


def api(path: str, payload: dict | None = None, timeout: int = 300) -> dict:
    data = None if payload is None else json.dumps(payload).encode("utf-8")
    req = urllib.request.Request(f"{OLLAMA_URL}{path}", data, {"Content-Type": "application/json"},
                                 method="GET" if payload is None else "POST")
    try:
        with urllib.request.urlopen(req, timeout=timeout) as resp:
            return json.loads(resp.read().decode("utf-8"))
    except urllib.error.HTTPError as exc:
        raise RuntimeError(f"{path} -> HTTP {exc.code}: {exc.read().decode('utf-8', errors='replace')}") from exc


def start_server(parallel: int = 8) -> None:
    """Start `ollama serve` pinned to GPU 0 (call once from the notebook)."""
    subprocess.run(["pkill", "-f", "ollama serve"], check=False)
    time.sleep(1)
    env = dict(os.environ, CUDA_VISIBLE_DEVICES="0", OLLAMA_NUM_PARALLEL=str(parallel), OLLAMA_MAX_LOADED_MODELS="1")
    open("ollama_parallel.txt", "w").write(str(parallel))   # lets the benchmark record the server setting
    with open("/tmp/ollama.log", "w") as log:
        subprocess.Popen(["ollama", "serve"], stdout=log, stderr=subprocess.STDOUT, env=env)
    for _ in range(60):
        try:
            api("/api/tags", timeout=3)
            return
        except Exception:
            time.sleep(1)
    raise RuntimeError("Ollama did not start; see /tmp/ollama.log")


def pull(model: str) -> None:
    print(f"pulling {model} ...")
    api("/api/pull", {"model": model, "name": model, "stream": False}, timeout=1800)   # both keys: older/newer Ollama


def gpu_info() -> tuple[str, int, int]:
    """(name, used MiB, total MiB) of GPU 0."""
    try:
        out = subprocess.check_output(
            ["nvidia-smi", "-i", "0", "--query-gpu=name,memory.used,memory.total", "--format=csv,noheader,nounits"],
            text=True).strip().splitlines()[0]
        name, used, total = [x.strip() for x in out.split(",")]
        return name, int(used), int(total)
    except Exception:
        return "no-gpu-visible", 0, 0


def unload(models) -> None:
    for m in models:
        try:
            api("/api/generate", {"model": m, "prompt": "", "keep_alive": 0}, timeout=60)
        except Exception:
            pass
    time.sleep(SLEEP)


def generate(model: str, prompt: str, num_predict: int = 200, num_ctx: int = 2048, keep_alive: str = "10m") -> dict:
    return api("/api/generate", {
        "model": model, "prompt": prompt, "stream": False, "keep_alive": keep_alive,
        "options": {"temperature": 0, "seed": 42, "num_predict": num_predict, "num_ctx": num_ctx},
    })


def arch_from_show(show: dict) -> dict:
    """Pull layer/head counts out of /api/show -> model_info (key prefix varies by architecture)."""
    mi = show.get("model_info", {})
    pick = lambda suffix: next((v for k, v in mi.items() if k.endswith(suffix)), None)
    return {"layers": pick(".block_count"), "heads": pick(".attention.head_count"),
            "kv_heads": pick(".attention.head_count_kv"), "embed": pick(".embedding_length"),
            "context_length": pick(".context_length")}


def kv_bytes_per_token(layers: int, heads: int, kv_heads: int, embed: int, bytes_per_value: int = 2) -> int:
    """K and V, per layer, per KV head: 2 * layers * kv_heads * head_dim * bytes (fp16 = 2)."""
    return 2 * layers * kv_heads * (embed // heads) * bytes_per_value


Overwriting task3_lib.py


In [6]:
%%writefile task3_bench.py
"""Task 3 benchmark: speed + VRAM per quantisation, context sweep, optional concurrency test, optional fp16.
    python task3_bench.py --no_concurrency      # server with parallel=1: q8_0 vs q4_K_M speed/VRAM + context sweep
    python task3_bench.py --concurrency_only    # server restarted with parallel=8: adds the concurrency test
    python task3_bench.py --fp16 --no_concurrency   # also benchmark fp16 (needs the fp16 model pulled)
Why two server settings: Ollama reserves KV cache for every parallel slot, which would inflate the VRAM numbers.
"""
import argparse
import json
import os
import threading
import time

from task3_lib import (SLEEP, FP16_MODEL, MODELS, PROMPTS, api, arch_from_show, generate, gpu_info,
                       kv_bytes_per_token, unload)

MAIN_CTX = 2048
SWEEP_CTXS = [512, 2048, 8192, 32768]
CONC_LEVELS = [1, 2, 4, 8]


def bench_model(model, idle_mib, all_models):
    unload(all_models)
    warm = generate(model, "Say 'ready' in one word.", 10, MAIN_CTX)
    load_s = round(warm.get("load_duration", 0) / 1e9, 3)
    time.sleep(SLEEP / 2)
    _, used, _ = gpu_info()
    ps = api("/api/ps").get("models", [])
    ps_row = next((m for m in ps if m.get("name", "").startswith(model.split(":")[0])), {})
    rows = []
    for i, prompt in enumerate(PROMPTS, 1):
        raw = generate(model, prompt, 200, MAIN_CTX)
        p_sec, e_sec = raw.get("prompt_eval_duration", 0) / 1e9, raw.get("eval_duration", 0) / 1e9
        row = {"id": i, "prompt": prompt, "response": raw.get("response", "").strip(),
               "prompt_tokens": raw.get("prompt_eval_count", 0), "prompt_s": round(p_sec, 4),
               "eval_tokens": raw.get("eval_count", 0), "eval_s": round(e_sec, 4),
               "gen_tps": round(raw.get("eval_count", 0) / e_sec, 2) if e_sec > 0 else 0.0}
        rows.append(row)
        print(f"[{model}] Q{i:02d}: {row['gen_tps']} tok/s ({row['eval_tokens']} tokens)")
    tp, ts = sum(r["prompt_tokens"] for r in rows), sum(r["prompt_s"] for r in rows)
    te, tes = sum(r["eval_tokens"] for r in rows), sum(r["eval_s"] for r in rows)
    return {"vram_used_mib": used, "vram_delta_mib": used - idle_mib, "cold_load_s": load_s,
            "api_ps_size_mib": round(ps_row.get("size", 0) / 2**20), "api_ps_size_vram_mib": round(ps_row.get("size_vram", 0) / 2**20),
            "weighted_prompt_tps": round(tp / ts, 2) if ts else 0.0, "weighted_gen_tps": round(te / tes, 2) if tes else 0.0,
            "mean_gen_tps": round(sum(r["gen_tps"] for r in rows) / len(rows), 2), "rows": rows}


def ctx_sweep(model, idle_mib, all_models):
    out = []
    for c in SWEEP_CTXS:
        unload(all_models)
        try:
            generate(model, "hi", 1, c)
            time.sleep(SLEEP / 2)
            _, used, _ = gpu_info()
            out.append({"num_ctx": c, "vram_used_mib": used, "vram_delta_mib": used - idle_mib})
            print(f"[{model}] num_ctx={c}: {used} MiB used")
        except Exception as exc:  # e.g. out of memory at very large context
            out.append({"num_ctx": c, "error": str(exc)[:200]})
            print(f"[{model}] num_ctx={c}: ERROR {str(exc)[:100]}")
    return out


def concurrency(model, all_models, ctx=MAIN_CTX, n_pred=128):
    unload(all_models)
    generate(model, "hi", 1, ctx)
    out = []
    for n in CONC_LEVELS:
        results, errors = [None] * n, []

        def work(i):
            try:
                results[i] = generate(model, PROMPTS[4], n_pred, ctx)
            except Exception as exc:
                errors.append(str(exc)[:150])

        threads = [threading.Thread(target=work, args=(i,)) for i in range(n)]
        t0 = time.time()
        [t.start() for t in threads]
        [t.join() for t in threads]
        wall = time.time() - t0
        ok = [r for r in results if r]
        toks = sum(r.get("eval_count", 0) for r in ok)
        per = [r["eval_count"] / (r["eval_duration"] / 1e9) for r in ok if r.get("eval_duration")]
        _, used, _ = gpu_info()
        row = {"concurrent_requests": n, "wall_s": round(wall, 2), "total_tokens": toks,
               "aggregate_tps": round(toks / wall, 1) if wall else 0,
               "mean_per_request_tps": round(sum(per) / len(per), 1) if per else 0,
               "vram_used_mib": used, "errors": errors}
        out.append(row)
        print(f"[concurrency {n}] aggregate {row['aggregate_tps']} tok/s | per request {row['mean_per_request_tps']} tok/s | wall {row['wall_s']}s")
    return out


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--fp16", action="store_true")
    ap.add_argument("--no_sweep", action="store_true")
    ap.add_argument("--no_concurrency", action="store_true")
    ap.add_argument("--concurrency_only", action="store_true", help="add the concurrency test to an existing task3_results.json (start the server with parallel=8 first)")
    a = ap.parse_args()
    models = MODELS + ([FP16_MODEL] if a.fp16 else [])
    parallel = open("ollama_parallel.txt").read().strip() if os.path.exists("ollama_parallel.txt") else "server default"

    if a.concurrency_only:   # server was restarted with more parallel slots; keep the single-slot results untouched
        results = json.load(open("task3_results.json", encoding="utf-8"))
        results["concurrency"] = {"model": models[1], "num_ctx": MAIN_CTX, "server_num_parallel": parallel,
                                  "rows": concurrency(models[1], models)}
        json.dump(results, open("task3_results.json", "w", encoding="utf-8"), indent=2, ensure_ascii=False)
        return

    generate(models[0], "hi", 1, MAIN_CTX)          # prime CUDA so the first model does not pay init cost
    unload(models)
    gpu_name, idle, total = gpu_info()
    tags = {m["name"]: m.get("size", 0) for m in api("/api/tags").get("models", [])}
    arch = arch_from_show(api("/api/show", {"model": models[-1], "name": models[-1]}))
    kv = kv_bytes_per_token(arch["layers"], arch["heads"], arch["kv_heads"], arch["embed"]) if all(arch[k] for k in ("layers", "heads", "kv_heads", "embed")) else None
    results = {"meta": {"gpu_name": gpu_name, "ollama_version": api("/api/version").get("version"),
                        "idle_vram_mib": idle, "total_vram_mib": total, "main_num_ctx": MAIN_CTX,
                        "ollama_num_parallel": parallel,
                        "kv_cache_type_assumed": "f16 (Ollama default)", "arch": arch, "kv_bytes_per_token": kv,
                        "vram_method": "nvidia-smi memory.used on GPU 0 (whole-GPU number, includes CUDA context)"},
               "models": {}, "model_file_bytes": {m: tags.get(m) for m in models}}
    for m in models:
        results["models"][m] = bench_model(m, idle, models)
        if not a.no_sweep:
            results["models"][m]["ctx_sweep"] = ctx_sweep(m, idle, models)
    if not a.no_concurrency:
        results["concurrency"] = {"model": models[1], "num_ctx": MAIN_CTX, "server_num_parallel": parallel, "rows": concurrency(models[1], models)}
    with open("task3_results.json", "w", encoding="utf-8") as fh:
        json.dump(results, fh, indent=2, ensure_ascii=False)

    print(f"\nGPU: {gpu_name} | ollama {results['meta']['ollama_version']} | num_ctx={MAIN_CTX} for the speed runs")
    print("| Model | Weighted gen tok/s | Mean gen tok/s | Prompt tok/s | Load (s) | VRAM used (whole GPU) | File size |")
    print("|---|---|---|---|---|---|---|")
    for m, d in results["models"].items():
        size = results["model_file_bytes"].get(m)
        print(f"| `{m}` | {d['weighted_gen_tps']} | {d['mean_gen_tps']} | {d['weighted_prompt_tps']} | {d['cold_load_s']} | "
              f"{d['vram_used_mib']} MiB | {round(size / 2**20) if size else 'n/a'} MiB |")
    if not a.no_sweep:
        print("\nVRAM vs context window (MiB, whole GPU):")
        print("| num_ctx | " + " | ".join(f"`{m.split(':')[1]}`" for m in models) + " |")
        print("|---|" + "---|" * len(models))
        for i, c in enumerate(SWEEP_CTXS):
            cells = [str(results["models"][m]["ctx_sweep"][i].get("vram_used_mib", "err")) for m in models]
            print(f"| {c} | " + " | ".join(cells) + " |")
    if kv:
        print(f"\nKV cache from model config: {kv} bytes/token ({kv / 1024:.1f} KiB), layers={arch['layers']}, kv_heads={arch['kv_heads']}, heads={arch['heads']}")


if __name__ == "__main__":
    main()


Overwriting task3_bench.py


In [7]:
%%writefile task3_quality.py
"""Task 3 output-quality checks + side-by-side review.
Every prompt gets a check, and the table says HOW it was checked:
  executed  = the model's code was run against tests (Python) or on a real SQLite database
  parsed    = JSON parsed and the VALUES checked
  counted   = sentence/bullet count checked
  keyword   = keywords present -> only a hint, READ the answer yourself
Writes task3_quality.json, task3_side_by_side.md (for your own observations) and task3_quality_table.md.
"""
import json
import os
import re
import sqlite3
import subprocess
import sys
import tempfile

FENCE = chr(96) * 3


def code_blocks(text: str) -> list[str]:
    return [b.strip() for b in re.findall(FENCE + r"(?:[a-zA-Z]*)\s*\n(.*?)" + FENCE, text, flags=re.S)]


def first_code(text: str) -> str:
    blocks = code_blocks(text)
    return blocks[0] if blocks else text.strip()


def run_with_tests(code: str, tests: str, timeout: int = 10) -> tuple[bool, str]:
    """Execute model code in a fresh interpreter (no __main__ side effects) and then the tests."""
    harness = f"src = {code!r}\nns = {{'__name__': 'model_code'}}\nexec(compile(src, '<model>', 'exec'), ns)\n{tests}\nprint('OK')\n"
    with tempfile.TemporaryDirectory() as d:
        path = os.path.join(d, "t.py")
        open(path, "w", encoding="utf-8").write(harness)
        try:
            p = subprocess.run([sys.executable, path], capture_output=True, text=True, timeout=timeout, cwd=d, stdin=subprocess.DEVNULL)
        except subprocess.TimeoutExpired:
            return False, "timeout"
    if p.returncode == 0 and p.stdout.strip().endswith("OK"):
        return True, "all tests passed"
    last = (p.stderr.strip().splitlines() or ["failed"])[-1]
    return False, last[:90]


def sentences(text: str) -> int:
    return len([s for s in re.split(r"(?<=[.!?])\s+", text.strip()) if s.strip()])


def check_q1(t):
    n, ok = sentences(t), "mutable" in t.lower()
    return (n == 3 and ok), "counted", f"{n} sentences, mentions mutability: {ok}"


def check_q2(t):
    tests = ("fn = ns['is_palindrome']\n"
             "assert bool(fn('A man, a plan, a canal: Panama')) is True\nassert bool(fn('No lemon, no melon')) is True\n"
             "assert bool(fn('Was it a car or a cat I saw?')) is True\nassert bool(fn('hello world')) is False\nassert bool(fn('race a car')) is False\n")
    ok, note = run_with_tests(first_code(t), tests)
    return ok, "executed", note


def check_q3(t):
    cleaned = re.sub(FENCE + r"(?:json)?", "", t, flags=re.I).strip()
    try:
        o = json.loads(cleaned)
    except Exception:
        return False, "parsed", "invalid JSON"
    if not isinstance(o, dict) or set(o) != {"name", "role", "years_exp"}:
        return False, "parsed", f"wrong keys {sorted(o) if isinstance(o, dict) else type(o).__name__}"
    good = (str(o["name"]).strip().lower() == "priya sharma" and "senior ml engineer" in str(o["role"]).lower()
            and str(o["years_exp"]).strip() == "6")
    return good, "parsed", "values correct" if good else f"wrong values {o}"


def check_q4(t):
    n = t.lower()
    right = "0.05" in n or "5 cents" in n or "five cents" in n
    wrong = re.search(r"ball (?:costs|is|=|would cost)\s*\$?\s*0?\.10\b", n) or "10 cents" in n and "5 cents" not in n
    return bool(right and not wrong), "keyword", "mentions $0.05" + (" but also asserts $0.10" if wrong else "") if right else "no $0.05 / 5 cents"


def check_q5(t):
    bullets = [l for l in t.splitlines() if re.match(r"^\s*([-*\u2022]|\d+[.)])\s+\S", l)]
    return len(bullets) == 3, "counted", f"{len(bullets)} bullet lines"


def check_q6(t):
    tests = ("fn = ns['average']\nassert fn([1, 2, 3]) == 2\n"
             "try:\n    fn([])\nexcept ZeroDivisionError:\n    raise AssertionError('still raises ZeroDivisionError on empty list')\nexcept Exception:\n    pass\n")
    ok, note = run_with_tests(first_code(t), tests)
    return ok, "executed", note


def check_q7(t):
    n = t.lower()
    mem = any(k in n for k in ("memory", "vram", "smaller", "size"))
    spd = any(k in n for k in ("speed", "faster", "latency", "throughput"))
    prec = any(k in n for k in ("precision", "bits", "accuracy", "quality"))
    return (mem and spd and prec), "keyword", f"memory={mem} speed={spd} precision/quality={prec}"


def check_q8(t):
    n = t.lower()
    informal = [w for w in ("asap", "thx", " u ", "hey") if w in f" {n} "]
    polite = any(k in n for k in ("please", "kindly", "would you", "could you"))
    ok = not informal and polite and "report" in n
    return ok, "keyword", f"informal words left: {informal or 'none'}; polite: {polite}"


def check_q9(t):
    rows = [("Ann", "Eng", 100), ("Bob", "Eng", 90), ("Cid", "Eng", 80), ("Dee", "Eng", 70), ("Eve", "Ops", 60),
            ("Fay", "Ops", 50), ("Gus", "Ops", 40), ("Hal", "Ops", 30), ("Ivy", "HR", 20)]
    expected = {"Ann", "Bob", "Cid", "Eve", "Fay", "Gus", "Ivy"}
    sql = first_code(t).strip().rstrip(";")
    con = sqlite3.connect(":memory:")
    con.execute("CREATE TABLE employees(id INTEGER, name TEXT, department TEXT, salary INTEGER)")
    con.executemany("INSERT INTO employees VALUES (?,?,?,?)", [(i, n, d, s) for i, (n, d, s) in enumerate(rows, 1)])
    try:
        cur = con.execute(sql)
        cols = [c[0].lower() for c in cur.description]
        data = cur.fetchall()
    except Exception as exc:
        return False, "executed", f"SQLite error: {str(exc)[:70]}"
    if "name" in cols:
        got = {r[cols.index("name")] for r in data}
    else:
        got = {c for r in data for c in r if isinstance(c, str)} & {r[0] for r in rows}
    return got == expected, "executed", f"returned {len(data)} rows; expected top-3 per department" + ("" if got == expected else f"; got names {sorted(got)}")


def check_q10(t):
    n = t.lower()
    terms = "precision" in n and "recall" in n
    parts = any(k in n for k in ("false positive", "false negative", "true positive", "false positives", "false negatives"))
    return (terms and parts), "keyword", f"precision&recall={terms}; confusion-matrix terms={parts}"


CHECKS = {1: check_q1, 2: check_q2, 3: check_q3, 4: check_q4, 5: check_q5, 6: check_q6, 7: check_q7, 8: check_q8, 9: check_q9, 10: check_q10}


def main():
    data = json.load(open("task3_results.json", encoding="utf-8"))
    models = list(data["models"])
    short = lambda m: m.split(":")[1]
    res = {m: {} for m in models}
    method = {}
    for m in models:
        for row in data["models"][m]["rows"]:
            ok, how, note = CHECKS[row["id"]](row["response"])
            res[m][row["id"]] = {"pass": bool(ok), "method": how, "note": note}
            method[row["id"]] = how
    lines = [f"SQLite {sqlite3.sqlite_version} used for Q9. Method: executed/parsed/counted = objective; keyword = hint only, read the answer.", "",
             "| Q | method | " + " | ".join(f"`{short(m)}`" for m in models) + " |", "|---|---|" + "---|" * len(models)]
    for q in range(1, 11):
        cells = [("PASS" if res[m][q]["pass"] else "FAIL") + f" ({res[m][q]['note']})" for m in models]
        lines.append(f"| Q{q:02d} | {method[q]} | " + " | ".join(cells) + " |")
    objective = [q for q in range(1, 11) if method[q] != "keyword"]
    lines.append("")
    for m in models:
        o = sum(res[m][q]["pass"] for q in objective)
        k = sum(res[m][q]["pass"] for q in range(1, 11) if method[q] == "keyword")
        lines.append(f"`{short(m)}`: objective checks {o}/{len(objective)} | keyword hints {k}/{10 - len(objective)} | total {o + k}/10")
    identical = sum(len({data['models'][m]['rows'][i]['response'] for m in models}) == 1 for i in range(10))
    lines.append(f"Identical responses across models: {identical}/10 (temperature 0, seed 42).")
    table = "\n".join(lines)
    print(table)
    open("task3_quality_table.md", "w", encoding="utf-8").write(table + "\n")
    json.dump({"checks": res, "method": method}, open("task3_quality.json", "w"), indent=2)

    md = ["# Task 3: side-by-side outputs (write YOUR observations under each prompt)", ""]
    for i in range(10):
        r0 = data["models"][models[0]]["rows"][i]
        md += [f"## Q{i + 1:02d}: {r0['prompt']}", ""]
        for m in models:
            r = data["models"][m]["rows"][i]
            c = res[m][r["id"]]
            md += [f"**{short(m)}** | {r['eval_tokens']} tokens | {r['gen_tps']} tok/s | check: {'PASS' if c['pass'] else 'FAIL'} ({c['method']}: {c['note']})", "",
                   "> " + r["response"][:1500].replace("\n", "\n> "), ""]
        md += ["**My observation:** ", ""]
    open("task3_side_by_side.md", "w", encoding="utf-8").write("\n".join(md))
    print("\nWrote task3_quality.json, task3_quality_table.md, task3_side_by_side.md")


if __name__ == "__main__":
    main()


Writing task3_quality.py


In [8]:
%%writefile task3_plan.py
"""Task 3 analysis helpers (offline, numbers only; the written explanations must be yours):
  A) KV cache size per token from the model config + cross-check against the measured context sweep
  B) bandwidth sanity check: why q4 decodes faster than q8
  C) 20-user / one-24GB-GPU memory and throughput calculator (ASSUMPTIONS are printed, change them with flags)
    python task3_plan.py --users 20 --ctx 2048 --gpu_gb 24 --bw 600
"""
import argparse
import json
import math
import os

from task3_lib import kv_bytes_per_token

p = argparse.ArgumentParser()
p.add_argument("--users", type=int, default=20)
p.add_argument("--ctx", type=int, default=2048, help="tokens per user (prompt + answer) kept in KV cache")
p.add_argument("--gpu_gb", type=float, default=24.0)
p.add_argument("--bw", type=float, default=600.0, help="GPU memory bandwidth GB/s (A10G ~600, L4 ~300, RTX 4090 ~1008)")
p.add_argument("--overhead_gb", type=float, default=1.5, help="CUDA context + runtime buffers")
p.add_argument("--headroom", type=float, default=0.90, help="fraction of GPU memory we allow ourselves to use")
a = p.parse_args()

res = json.load(open("task3_results.json", encoding="utf-8")) if os.path.exists("task3_results.json") else None
meta = (res or {}).get("meta", {})
kv = meta.get("kv_bytes_per_token") or kv_bytes_per_token(28, 12, 2, 1536)   # Qwen2.5-1.5B fallback
src = "from /api/show model config" if meta.get("kv_bytes_per_token") else "fallback: Qwen2.5-1.5B config (28 layers, 2 KV heads, head_dim 128)"
GiB = 2**30

print("=== A) KV cache per token ===")
print(f"KV bytes/token = 2 (K and V) x layers x kv_heads x head_dim x 2 bytes (fp16) = {kv} B = {kv / 1024:.1f} KiB   [{src}]")
print("| context | KV cache (MiB) |")
print("|---|---|")
for c in (512, 2048, 4096, 8192, 32768):
    print(f"| {c} | {kv * c / 2**20:.0f} |")
if res:
    m0 = list(res["models"])[1]
    sweep = [s for s in res["models"][m0].get("ctx_sweep", []) if "vram_used_mib" in s]
    if len(sweep) > 1:
        base = sweep[0]
        print(f"\nCross-check on `{m0}` (VRAM growth vs num_ctx={base['num_ctx']}; measured includes compute buffers, so >= predicted):")
        print("| num_ctx | measured growth (MiB) | predicted KV growth (MiB) |")
        print("|---|---|---|")
        for s in sweep[1:]:
            print(f"| {s['num_ctx']} | {s['vram_used_mib'] - base['vram_used_mib']} | {kv * (s['num_ctx'] - base['num_ctx']) / 2**20:.0f} |")

effs = []
if res and res.get("model_file_bytes"):
    print("\n=== B) Why quantisation speeds up decoding (memory-bandwidth view) ===")
    print("Each generated token must read (about) all weights once, so tok/s x file size ~ bandwidth actually used.")
    print("| model | file size GB | gen tok/s | implied bandwidth GB/s |")
    print("|---|---|---|---|")
    for m, d in res["models"].items():
        size = res["model_file_bytes"].get(m)
        if size and d["weighted_gen_tps"]:
            bwu = size / 1e9 * d["weighted_gen_tps"]
            effs.append(bwu)
            print(f"| {m.split(':')[1]} | {size / 1e9:.2f} | {d['weighted_gen_tps']} | {bwu:.0f} |")
    print(f"Compare with the GPU's peak memory bandwidth ({meta.get('gpu_name', 'GPU')}: look up the published spec, e.g. Tesla T4 = 320 GB/s).")
    print("The two implied bandwidths will not be identical (dequantisation work and per-token overhead differ), but if both are a")
    print("sizeable fraction of the GPU peak, decoding is memory-bound, so fewer bytes per weight gives more tokens/s.")

print("\n=== C) Serving", a.users, "users on one", a.gpu_gb, "GB GPU ===")
eff = 0.5
print(f"ASSUMPTIONS: {a.ctx} tokens of KV cache per user (all {a.users} active at once), fp16 KV cache, {a.overhead_gb} GB runtime overhead, "
      f"{a.headroom:.0%} of GPU memory usable, GPU bandwidth {a.bw:.0f} GB/s at {eff:.0%} efficiency (decode is memory-bound; prefill ignored).")
weights_q4 = (res or {}).get("model_file_bytes", {}).get("qwen2.5:1.5b-instruct-q4_K_M")
cands = [  # (name, weights GB, kv bytes/token, source)
    ("Qwen2.5-1.5B q4_K_M", (weights_q4 or 0.99e9) / 1e9, kv, "measured here" if weights_q4 else "approx."),
    ("Qwen2.5-7B q4_K_M", 4.7, kv_bytes_per_token(28, 28, 4, 3584), "assumed sizes"),
    ("Llama-3.1-8B q4_K_M", 4.9, kv_bytes_per_token(32, 32, 8, 4096), "assumed sizes"),
    ("Llama-3.1-8B fp16", 16.0, kv_bytes_per_token(32, 32, 8, 4096), "assumed sizes"),
]
usable = a.gpu_gb * a.headroom
print(f"| model ({a.users} users, ctx {a.ctx}) | weights GB | KV/token KiB | KV total GB | total GB (+overhead) | fits in {usable:.1f} GB? | max users at this ctx | est. tok/s per user | est. total tok/s |")
print("|---|---|---|---|---|---|---|---|---|")
for name, w, kvb, how in cands:
    kv_total = a.users * a.ctx * kvb / 1e9
    total = w + kv_total + a.overhead_gb
    max_users = max(0, math.floor((usable - w - a.overhead_gb) * 1e9 / (a.ctx * kvb)))
    step = (w + kv_total) / (a.bw * eff)       # seconds per decode step, all users advance one token
    print(f"| {name} ({how}) | {w:.1f} | {kvb / 1024:.0f} | {kv_total:.1f} | {total:.1f} | {'yes' if total <= usable else 'NO'} | {max_users} | {1 / step:.0f} | {a.users / step:.0f} |")
print("\nHow the numbers are made: total = weights + users x ctx x KV/token + overhead; one decode step reads weights + all KV once,")
print("so step time = bytes / (bandwidth x efficiency) and every user gets 1/step tokens per second (continuous batching assumed).")
print("These are estimates from stated assumptions, NOT measurements. If you ran the concurrency test, compare it with them.")
if res and res.get("concurrency"):
    print("\nMeasured on this GPU (concurrency test):")
    print("| concurrent requests | aggregate tok/s | mean per-request tok/s |")
    print("|---|---|---|")
    for r in res["concurrency"]["rows"]:
        print(f"| {r['concurrent_requests']} | {r['aggregate_tps']} | {r['mean_per_request_tps']} |")


Writing task3_plan.py


## Phase 1: server with 1 slot -> speed, VRAM, context sweep

In [9]:
from task3_lib import start_server, pull, MODELS, FP16_MODEL

start_server(parallel=1)   # pinned to GPU 0
for m in MODELS:
    pull(m)
# Optional 16-bit baseline (about 3 GB download): uncomment this line AND use the fp16 benchmark cell below
# pull(FP16_MODEL)

pulling qwen2.5:1.5b-instruct-q8_0 ...
pulling qwen2.5:1.5b-instruct-q4_K_M ...


In [10]:
!python task3_bench.py --no_concurrency

[qwen2.5:1.5b-instruct-q8_0] Q01: 89.53 tok/s (42 tokens)
[qwen2.5:1.5b-instruct-q8_0] Q02: 100.8 tok/s (200 tokens)
[qwen2.5:1.5b-instruct-q8_0] Q03: 103.7 tok/s (33 tokens)
[qwen2.5:1.5b-instruct-q8_0] Q04: 100.81 tok/s (200 tokens)
[qwen2.5:1.5b-instruct-q8_0] Q05: 100.65 tok/s (154 tokens)
[qwen2.5:1.5b-instruct-q8_0] Q06: 100.05 tok/s (183 tokens)
[qwen2.5:1.5b-instruct-q8_0] Q07: 99.9 tok/s (200 tokens)
[qwen2.5:1.5b-instruct-q8_0] Q08: 108.33 tok/s (15 tokens)
[qwen2.5:1.5b-instruct-q8_0] Q09: 100.09 tok/s (200 tokens)
[qwen2.5:1.5b-instruct-q8_0] Q10: 100.03 tok/s (200 tokens)
[qwen2.5:1.5b-instruct-q8_0] num_ctx=512: 1771 MiB used
[qwen2.5:1.5b-instruct-q8_0] num_ctx=2048: 1815 MiB used
[qwen2.5:1.5b-instruct-q8_0] num_ctx=8192: 2061 MiB used
[qwen2.5:1.5b-instruct-q8_0] num_ctx=32768: 2781 MiB used
[qwen2.5:1.5b-instruct-q4_K_M] Q01: 108.46 tok/s (43 tokens)
[qwen2.5:1.5b-instruct-q4_K_M] Q02: 132.95 tok/s (200 tokens)
[qwen2.5:1.5b-instruct-q4_K_M] Q03: 138.34 tok/s (34 toke

In [11]:
# OPTIONAL fp16 run (replaces the results file with a 3-model version; needs pull(FP16_MODEL) above)
# !python task3_bench.py --fp16 --no_concurrency

## Phase 2: restart with 8 slots -> concurrency test (adds to the same results file)

In [12]:
from task3_lib import start_server
start_server(parallel=8)

In [13]:
!python task3_bench.py --concurrency_only

[concurrency 1] aggregate 118.6 tok/s | per request 122.6 tok/s | wall 0.76s
[concurrency 2] aggregate 210.3 tok/s | per request 108.4 tok/s | wall 0.69s
[concurrency 4] aggregate 281.4 tok/s | per request 74.1 tok/s | wall 1.01s
[concurrency 8] aggregate 357.8 tok/s | per request 50.4 tok/s | wall 1.49s


## Quality: executed tests + side-by-side outputs

In [14]:
!python task3_quality.py

SQLite 3.45.1 used for Q9. Method: executed/parsed/counted = objective; keyword = hint only, read the answer.

| Q | method | `1.5b-instruct-q8_0` | `1.5b-instruct-q4_K_M` |
|---|---|---|---|
| Q01 | counted | FAIL (2 sentences, mentions mutability: True) | PASS (3 sentences, mentions mutability: True) |
| Q02 | executed | FAIL (SyntaxError: unterminated string literal (detected at line 1)) | FAIL (SyntaxError: invalid syntax) |
| Q03 | parsed | PASS (values correct) | FAIL (wrong values {'name': 'Priya Sharma', 'role': 'Senior ML Engineer', 'years_exp': '6 years'}) |
| Q04 | keyword | PASS (mentions $0.05) | PASS (mentions $0.05) |
| Q05 | counted | PASS (3 bullet lines) | PASS (3 bullet lines) |
| Q06 | executed | PASS (all tests passed) | PASS (all tests passed) |
| Q07 | keyword | PASS (memory=True speed=True precision/quality=True) | FAIL (memory=True speed=False precision/quality=True) |
| Q08 | keyword | PASS (informal words left: none; polite: True) | PASS (informal words left:

In [15]:
print(open('task3_side_by_side.md', encoding='utf-8').read()[:12000])

# Task 3: side-by-side outputs (write YOUR observations under each prompt)

## Q01: Explain the difference between a list and a tuple in Python in 3 concise sentences.

**1.5b-instruct-q8_0** | 42 tokens | 89.53 tok/s | check: FAIL (counted: 2 sentences, mentions mutability: True)

> A list is a mutable sequence type, allowing changes to its elements, while a tuple is an immutable sequence type, meaning its elements cannot be changed. Lists are defined with square brackets [], tuples with parentheses ().

**1.5b-instruct-q4_K_M** | 43 tokens | 108.46 tok/s | check: PASS (counted: 3 sentences, mentions mutability: True)

> A list is a mutable collection of items, while a tuple is an immutable collection of items. Lists can be changed after creation, tuples cannot. Elements in a list are accessed by index, tuples by position.

**My observation:** 

## Q02: Write a Python function named is_palindrome(s) returning True if s is a palindrome, ignoring spaces, punctuation, and case.

**1.5b-i

## KV cache, bandwidth check, and the 20-user calculator

In [16]:
!python task3_plan.py --users 20 --ctx 2048 --gpu_gb 24 --bw 600

=== A) KV cache per token ===
KV bytes/token = 2 (K and V) x layers x kv_heads x head_dim x 2 bytes (fp16) = 28672 B = 28.0 KiB   [from /api/show model config]
| context | KV cache (MiB) |
|---|---|
| 512 | 14 |
| 2048 | 56 |
| 4096 | 112 |
| 8192 | 224 |
| 32768 | 896 |

Cross-check on `qwen2.5:1.5b-instruct-q4_K_M` (VRAM growth vs num_ctx=512; measured includes compute buffers, so >= predicted):
| num_ctx | measured growth (MiB) | predicted KV growth (MiB) |
|---|---|---|
| 2048 | 44 | 42 |
| 8192 | 290 | 210 |
| 32768 | 1010 | 882 |

=== B) Why quantisation speeds up decoding (memory-bandwidth view) ===
Each generated token must read (about) all weights once, so tok/s x file size ~ bandwidth actually used.
| model | file size GB | gen tok/s | implied bandwidth GB/s |
|---|---|---|---|
| 1.5b-instruct-q8_0 | 1.65 | 100.12 | 165 |
| 1.5b-instruct-q4_K_M | 0.99 | 131.42 | 130 |
Compare with the GPU's peak memory bandwidth (Tesla T4: look up the published spec, e.g. Tesla T4 = 320 GB/s)

## Download everything

In [17]:
!zip -q task3_outputs.zip task3_results.json task3_quality.json task3_quality_table.md task3_side_by_side.md task3_lib.py task3_bench.py task3_quality.py task3_plan.py
!ls -la task3_outputs.zip

-rw-r--r-- 1 root root 24297 Nov 22 04:48 task3_outputs.zip


## Task 3 Report: Local LLM Inference & Optimisation (Ollama, Qwen2.5-1.5B)

### 1. Hardware & Setup
- **GPU:** NVIDIA Tesla T4 (320 GB/s peak memory bandwidth), pinned to GPU 0
- **Runtime:** Ollama `0.40.17` via HTTP API (`num_ctx=2048` for main speed/quality runs)
- **Two-phase design:** Phase 1 ran with `OLLAMA_NUM_PARALLEL=1` so multi-slot KV cache reservations did not inflate single-user VRAM; Phase 2 restarted with `OLLAMA_NUM_PARALLEL=8` strictly for concurrency testing.

---

### 2. Quantisation: Speed, VRAM & Memory Bandwidth (`q8_0` vs `q4_K_M`)

| Model | Weighted Gen (tok/s) | Prompt (tok/s) | Load (s) | VRAM (MiB) | File Size (MiB) | Implied BW (GB/s) |
| :--- | :---: | :---: | :---: | :---: | :---: | :---: |
| `qwen2.5:1.5b-instruct-q8_0` | 100.12 | 2,757.2 | 3.013 | 1,815 | 1,570 | 165 |
| `qwen2.5:1.5b-instruct-q4_K_M` | **131.42** | **2,966.24** | **2.537** | **1,185** | **940** | 130 |

- **Key trade-off:** Switching from `q8_0` to `q4_K_M` increased generation speed by **+31.3%** and reduced VRAM usage by **34.7% (-630 MiB)** and disk footprint by **40%**.
- **Why quantisation speeds up decoding:** Autoregressive decoding reads nearly all model weights once per generated token. Both models hit a large fraction of the T4's 320 GB/s peak bandwidth (165 GB/s and 130 GB/s), confirming that decoding is **memory-bandwidth bound**—fewer bytes per weight directly translates to more tokens/second.

---

### 3. KV Cache & Context Window Scaling
From `/api/show`, Qwen2.5-1.5B has `layers=28`, `heads=12`, `kv_heads=2` (GQA), giving **28.0 KiB/token** in fp16 KV cache.

| `num_ctx` | `q8_0` VRAM (MiB) | `q4_K_M` VRAM (MiB) | `q4_K_M` Measured Growth | Predicted KV Growth |
| :---: | :---: | :---: | :---: | :---: |
| 512 | 1,771 | 1,141 | — | — |
| 2,048 | 1,815 | 1,185 | +44 MiB | +42 MiB |
| 8,192 | 2,061 | 1,431 | +290 MiB | +210 MiB |
| 32,768 | 2,781 | 2,151 | +1,010 MiB | +882 MiB |

Measured VRAM grows linearly with context length and stays slightly above the KV-only formula because of attention/compute buffers.

---

### 4. Concurrency Scaling (`q4_K_M`, 8 Parallel Slots)

| Concurrent Requests | Aggregate (tok/s) | Mean Per-Request (tok/s) | Wall Time (s) |
| :---: | :---: | :---: | :---: |
| 1 | 118.6 | 122.6 | 0.76 |
| 2 | 210.3 | 108.4 | 0.69 |
| 4 | 281.4 | 74.1 | 1.01 |
| 8 | **357.8** | 50.4 | 1.49 |

Batching 8 concurrent requests tripled aggregate throughput (**118.6 → 357.8 tok/s**) because model weights are read once per batched decode step, while per-user speed remained comfortable at **50.4 tok/s**.

---

### 5. Quality Comparison (`q8_0` vs `q4_K_M`)
- **Automated scores:** `q8_0` scored **7/10** (3/6 objective, 4/4 keyword) vs. `q4_K_M` at **6/10** (3/6 objective, 3/4 keyword), with **0/10** identical responses at temperature 0.
- **Truncation artifact:** The benchmark's `num_predict=200` cap cut off Q02, Q04, Q07, Q09, and Q10 mid-output—causing `SyntaxError` failures in Q02 and Q09 even where the core logic was partially or fully right.
- **Notable differences:**
  - **Q02 (Palindrome):** `q8_0` wrote correct logic (`re.sub` + `.lower()`) before truncation; `q4_K_M` forgot `.lower()`, breaking case-insensitivity.
  - **Q03 (JSON):** `q8_0` extracted `"6"` cleanly; `q4_K_M` included the unit (`"6 years"`).
  - **Q05 & Q07 (Explanations):** `q8_0` gave more specific technical detail; `q4_K_M` was shorter and omitted speed in Q07.
  - **Q01 (Sentence count):** `q4_K_M` followed the 3-sentence limit while `q8_0` wrote 2 sentences.

---

### 6. Serving 20 Concurrent Users on One 24 GB GPU (`ctx=2048`)
Assuming 21.6 GB usable VRAM (90% headroom), 1.5 GB runtime overhead, fp16 KV cache, and 600 GB/s bandwidth at 50% efficiency:
- **`Qwen2.5-1.5B q4_K_M` (3.7 GB total):** Fits easily (up to 325 users; ~139 tok/s/user).
- **`Qwen2.5-7B q4_K_M` (8.5 GB total):** Fits comfortably (up to 131 users; ~43 tok/s/user)—best balance of quality and capacity.
- **`Llama-3.1-8B q4_K_M` (11.8 GB total):** Fits (up to 56 users; ~29 tok/s/user), though its larger KV cache (128 KiB/token) limits max concurrency.
- **`Llama-3.1-8B fp16` (22.9 GB total):** **Does not fit** (max 15 users).

**Takeaway:** For production serving on a single 24 GB GPU, 4-bit quantisation (`q4_K_M`) is essential—it frees enough VRAM for concurrent KV caches and boosts decode throughput with only minor quality loss compared to `q8_0`.